In [7]:
from dotenv import load_dotenv

load_dotenv()

True

In [8]:
import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    
    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__


## Local MCP server

In [9]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "local_server": {
                "transport": "stdio",
                "command": "python",
                "args": ["resources/2.1_mcp_server.py"],
            }
    }
)

In [10]:
# get tools
tools = await client.get_tools()

# get resources
resources = await client.get_resources("local_server")

# get prompts
prompt = await client.get_prompt("local_server", "prompt")
prompt = prompt[0].content

In [11]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    tools=tools,
    system_prompt=prompt
)

In [12]:
from langchain.messages import HumanMessage

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Tell me about the langchain-mcp-adapters library")]}
)

In [13]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Tell me about the langchain-mcp-adapters library', additional_kwargs={}, response_metadata={}, id='f4aa9560-ccc5-48f7-90b5-d4db09bb9758'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_web', 'arguments': '{"query": "what is langchain-mcp-adapters"}'}, '__gemini_function_call_thought_signatures__': {'call_7010029482189874929': 'EnMKcQFpFH0TjjU9wQgwgZpmLhMNObsQWA/2OlciGN4LuwlF/kDFO2NSNB0YgbVlWPeHbCidaI2TJKfqAjQ/7pmMV/Hn6XOE2SKPvuhyjonAO/xA+EHaVC714a4HFxkIjgJIly3DMgRoZVTysojSwSU0QgXc'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1cfcf-ec4d-7472-8766-bcd14c030089-0', tool_calls=[{'name': 'search_web', 'args': {'query': 'what is langchain-mcp-adapters'}, 'id': 'call_7010029482189874929', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 199, 'output_tokens': 23, 'total_token

## Online MCP

In [14]:
client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "uv",
            "args": [
                "run",
                "python",
                "-m",
                "mcp_server_time",
                "--local-timezone=America/New_York"
            ]
        }
    }
)

tools = await client.get_tools()

In [15]:
agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    tools=tools,
)

In [16]:
question = HumanMessage(content="What time is it?")

response = await agent.ainvoke(
    {"messages": [question]}
)

pprint(response)

{'messages': [HumanMessage(content='What time is it?', additional_kwargs={}, response_metadata={}, id='2f90ea54-8b62-48df-8a74-7365f3d8c883'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_current_time', 'arguments': '{"timezone": "America/New_York"}'}, '__gemini_function_call_thought_signatures__': {'call_5627359385414585547': 'EnMKcQFpFH0TBU+U9w0GTJE3j4127uX5TLsw881HveJ/X7Iq4mPNYnkLtuFo5y9PIib+7TpHA/WutQsLvpT/HFzQUnz7POjSG+VLWKnmaMZ/B4CmouJx6LS0+cdHHzijWlO5+0dAclTOr32uzOZuWxD8ucyr'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1cfd0-31e6-7492-b838-bf1c12f4f448-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'America/New_York'}, 'id': 'call_5627359385414585547', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 284, 'output_tokens': 22, 'total_tokens': 306, 'input_token_details': {'cache_re